# Evaluation & Inference

This notebook loads the best model checkpoint from training, visualises learning curves, generates a confusion matrix, and runs inference on the test set.

In [ ]:
import pickle
from pathlib import Path
import tensorflow as tf
from src.pipeline.evaluation import evaluate_model, plot_training_history
from src.pipeline.data_pipeline import get_train_val_test

# Load processed data
_, _, test_ds = get_train_val_test(image_size=(150, 150), batch_size=32)
class_names = ['cat', 'dog', 'other']

# Load the best checkpoint from training
checkpoint_path = "models_output/checkpoints/best.keras"
if Path(checkpoint_path).exists():
    model = tf.keras.models.load_model(checkpoint_path)
    print(f"Loaded model from {checkpoint_path}")
else:
    from src.pipeline.model import build_cnn_model, compile_model
    model = build_cnn_model(input_shape=(150, 150, 3), num_classes=3)
    compile_model(model)
    print("Checkpoint not found — using untrained model. Train first via 03_model_training.ipynb")

# Plot training curves from persisted history
hist_path = Path("reports/figures/training_history.pkl")
if hist_path.exists():
    with open(hist_path, "rb") as f:
        history_data = pickle.load(f)
    class FakeHistory:
        pass
    fake_hist = FakeHistory()
    fake_hist.history = history_data
    plot_training_history(fake_hist)
else:
    print("Training history not found. Run 03_model_training.ipynb first.")

# Run full evaluation on test set
results = evaluate_model(model, test_ds, class_names=class_names)
print(f'\nTest Accuracy: {results["accuracy"]:.4f}')